In [0]:
%run "../0_common/01. env-config"

In [0]:
catalog_name, landing_folder_path

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
v_batch_id

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

circuit_schema = StructType(
    [
        StructField("circuitId", StringType()),
        StructField("url", StringType()),
        StructField("circuitName", StringType()),
        StructField("lat", DoubleType()),
        StructField("long", DoubleType()),
        StructField("locality", StringType()),
        StructField("country", StringType())
    ]
)

In [0]:
df = (
    spark.read
    .format('csv')
    .option("header","true")
    .schema(circuit_schema)
    .load(source_file)
)

In [0]:
display(df.head(5))

In [0]:
import pyspark.sql.functions as F
df = (
    df
    .withColumn("ingestion_timestamp",F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))
)

 

In [0]:
display(df.head(5))

In [0]:
df = df.withColumn("batch_id", F.lit(v_batch_id))

In [0]:
(df
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("batch_id")
    .option("replaceWhere",f"batch_id = '{v_batch_id}'")
    .saveAsTable(table_name)
)

In [0]:
display(spark.table(table_name))